# More about time series

Astropy has a class called `TimeSeries` dedicated to representing time series.

We'll start by importing that, then read in more AAVSO data on DY Her and make a time series from it.

In [ ]:
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.table import Table
from astropy.timeseries import TimeSeries
from astropy.time import Time
from astropy import units as u

%matplotlib widget
from matplotlib import pyplot as plt

## More DY Her data -- all AAVSO data for DY Her in V band

We'll begin nearby reading this and plotting it.

In [ ]:
dy_her_V = Table.read('aavsodata_dy_her_V_only.csv')

In [ ]:
plt.figure()
plt.plot(dy_her_V['JD'], dy_her_V['Magnitude'])
plt.grid()

### Making a folded (phased) light curve

We begin by creating a time series. It is essentially a `Table` with a special `time` property.

It is often necessary to create a `Time` object first.

In [ ]:
times = Time(dy_her_V['JD'], format='jd', scale='utc')
time_series = TimeSeries(time=times, data=dy_her_V)

In [ ]:
time_series

There is a fold method for creating a phased light curve. The period for this star from VSX is below.

In [ ]:
period = 0.1486309 * u.day

folded_dy_her = time_series.fold(period=period, normalize_phase=True)

This turns the `time` into a number from 0 to 1.

In [ ]:
folded_dy_her.time

In [ ]:
plt.figure()
plt.plot(folded_dy_her.time, folded_dy_her['Magnitude'])
plt.grid()

In [ ]:
plt.figure()

# The "." below is shorthand for a marker that is a small dot

plt.plot(folded_dy_her.time, folded_dy_her['Magnitude'], '.')
plt.grid()

In [ ]:
dy_her_coord = SkyCoord.from_name('dy her')

hjd = times.light_travel_time(dy_her_coord, kind='heliocentric', location=EarthLocation(lon=0, lat=0))

In [ ]:
hjd = (times + hjd).utc

In [ ]:
hjd

In [ ]:
recent = hjd.jd > 2460000

In [ ]:
new_time_series = TimeSeries(time=hjd[recent], data=dy_her_V[recent])

In [ ]:
epoch = Time(2433439.4865, format='jd', scale='utc')
new_folded_dy_her = new_time_series.fold(period=period, normalize_phase=True, epoch_time=epoch)

In [ ]:
plt.figure()

# The "." below is shorthand for a marker that is a small dot

plt.plot(new_folded_dy_her.time, new_folded_dy_her['Magnitude'], '.', color='blue')
plt.plot(new_folded_dy_her.time + 1, new_folded_dy_her['Magnitude'], '.', color='blue')
plt.ylim(reversed(plt.ylim()))
plt.grid()